# 🛣️ NETRA — Pothole Detection Training Pipeline
## YOLOv8-seg Fine-Tuning on Pothole Dataset (Google Colab)

This notebook trains a YOLOv8 instance-segmentation model on pothole data.

**Steps:**
1. Setup GPU & install dependencies
2. Download dataset (Roboflow — no API key needed)
3. Train YOLOv8-seg
4. Evaluate & export model
5. Download `best.pt` for NETRA pipeline

---
**⚠️ Before running:** Go to **Runtime → Change runtime type → GPU (T4)** then **Runtime → Run all**


## Step 1: Setup Environment


In [ ]:
# Check GPU
!nvidia-smi


In [ ]:
# Install dependencies
!pip install -U ultralytics roboflow --quiet
print("✅ Packages installed")


In [ ]:
import os, shutil, glob
from pathlib import Path
from ultralytics import YOLO
print("✅ Imports ready")


## Step 2: Download Pothole Dataset

Downloads a public pothole detection dataset directly — **no API keys needed**.


In [ ]:
# ══════════════════════════════════════════════════════════════
#  OPTION A: Use YOUR OWN Roboflow API key (RECOMMENDED)
#
#  1. Go to https://app.roboflow.com → sign up (free)
#  2. Account → Settings → Roboflow API Key → copy it
#  3. Search "pothole detection" on https://universe.roboflow.com
#  4. Pick a dataset and click "Download this Dataset" → YOLOv8 format
#  5. It gives you a code snippet — paste below.
#
#  OR: just set your API key below and the default dataset will work.
# ══════════════════════════════════════════════════════════════

ROBOFLOW_API_KEY = ""  # ← PASTE YOUR KEY HERE (free signup at roboflow.com)

YOLO_DIR = "/content/pothole_dataset"

if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("atikul-islam-sajib").project("pothole-detection-yolov8-3")
    dataset = project.version(1).download("yolov8", location=YOLO_DIR)
    print("✅ Downloaded via Roboflow API")
else:
    # ══════════════════════════════════════════════════════════════
    #  OPTION B: Direct download (no API key needed)
    #  Uses a public pothole dataset hosted on GitHub releases
    # ══════════════════════════════════════════════════════════════
    import urllib.request, zipfile

    # Pothole dataset in YOLO format from a public GitHub release
    urls = [
        "https://github.com/atikul-islam-sajib/Research-Dataset/raw/main/pothole.zip",
    ]

    zip_path = "/content/pothole_dataset.zip"
    downloaded = False

    for url in urls:
        try:
            print(f"⬇️  Trying: {url.split('/')[-1]}...")
            urllib.request.urlretrieve(url, zip_path)
            size_mb = os.path.getsize(zip_path) / 1e6
            if size_mb > 1:  # valid zip must be > 1MB
                print(f"   Downloaded: {size_mb:.1f} MB ✅")
                downloaded = True
                break
            else:
                print(f"   Too small ({size_mb:.1f} MB) — trying next...")
                os.remove(zip_path)
        except Exception as e:
            print(f"   Failed: {e}")

    if not downloaded:
        print("\n❌ Auto-download failed. Manual steps:")
        print("   1. Go to https://universe.roboflow.com/search?q=pothole+detection")
        print("   2. Pick a dataset → Download → YOLOv8 → Get download code")
        print("   3. Paste the code in this cell and re-run")
        raise RuntimeError("Dataset download failed — see instructions above")

    print("📦 Extracting...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(YOLO_DIR)
    os.remove(zip_path)
    print("   Extracted ✅")

# ── Find data.yaml ────────────────────────────────────────────────
import yaml
yaml_candidates = glob.glob(f"{YOLO_DIR}/**/data.yaml", recursive=True)

if yaml_candidates:
    yaml_path = yaml_candidates[0]
    with open(yaml_path) as f:
        data_cfg = yaml.safe_load(f)
    data_cfg["path"] = str(Path(yaml_path).parent)
    with open(yaml_path, "w") as f:
        yaml.dump(data_cfg, f, sort_keys=False)
else:
    # Create data.yaml if missing
    yaml_path = f"{YOLO_DIR}/data.yaml"
    val_name = "valid" if os.path.exists(f"{YOLO_DIR}/valid") else "val"
    data_cfg = {
        "path": YOLO_DIR,
        "train": "train/images",
        "val": f"{val_name}/images",
        "names": {0: "pothole"},
        "nc": 1,
    }
    with open(yaml_path, "w") as f:
        yaml.dump(data_cfg, f, sort_keys=False)

# ── Show stats ────────────────────────────────────────────────────
base = str(Path(yaml_path).parent)
print(f"\n📂 Dataset: {base}")
print(f"   Classes: {data_cfg.get('names', {})}")
total = 0
for s in ("train", "valid", "val", "test"):
    d = f"{base}/{s}/images"
    if os.path.exists(d):
        n = len(os.listdir(d))
        total += n
        print(f"   {s}: {n} images")
print(f"   Total: {total} images")
assert total > 0, "❌ No images — dataset download failed"
print("\n✅ Dataset ready!")


In [ ]:
# Visualise a few training samples
import cv2, matplotlib.pyplot as plt

base = str(Path(yaml_path).parent)
train_img_dir = None
for d in ("train/images", "valid/images", "val/images"):
    p = f"{base}/{d}"
    if os.path.exists(p) and len(os.listdir(p)) > 0:
        train_img_dir = p
        break

if train_img_dir:
    sample_imgs = glob.glob(f"{train_img_dir}/*")[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, img_path in zip(axes.flat, sample_imgs):
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        lbl = img_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        if os.path.exists(lbl):
            for line in open(lbl):
                parts = line.strip().split()
                if len(parts) >= 5:
                    _, xc, yc, bw, bh = [float(x) for x in parts[:5]]
                    x1, y1 = int((xc-bw/2)*w), int((yc-bh/2)*h)
                    x2, y2 = int((xc+bw/2)*w), int((yc+bh/2)*h)
                    cv2.rectangle(img, (x1,y1), (x2,y2), (255,0,0), 2)
        ax.imshow(img); ax.axis("off")
    plt.suptitle("Training Samples", fontsize=14)
    plt.tight_layout(); plt.show()
else:
    print("⚠️ No training images found to visualize")


## Step 3: Train YOLOv8-seg

**Model options** (change `MODEL_SIZE` below):
- `yolov8n-seg.pt` — nano: fastest, lowest accuracy
- `yolov8s-seg.pt` — **small: best speed/accuracy trade-off** ← default
- `yolov8m-seg.pt` — medium: best accuracy, slower


In [ ]:
# ══════════════════════════════════════════════════════════
#  TRAINING CONFIG — change these as needed
# ══════════════════════════════════════════════════════════
MODEL_SIZE = "yolov8s-seg.pt"
EPOCHS     = 150
BATCH_SIZE = 16              # reduce to 8 if OOM
IMG_SIZE   = 640
PATIENCE   = 50
# ══════════════════════════════════════════════════════════


In [ ]:
# Load pretrained YOLOv8-seg and train
model = YOLO(MODEL_SIZE)

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    project="/content/runs",
    name="netra-pothole",
    plots=True,

    # Augmentation (road/dashcam scenes)
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    erasing=0.4,

    # LR schedule
    cos_lr=True,
    lr0=0.01,
    lrf=0.005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    weight_decay=0.0005,
    optimizer="SGD",

    # Mosaic warm-down (disable last 15 epochs → +1-2 mAP)
    close_mosaic=15,

    # Saving
    save_period=25,
    patience=PATIENCE,
    verbose=True,
)

run_dir = "/content/runs/netra-pothole"
print("\n✅ Training complete!")


## Step 4: Evaluate Results


In [ ]:
# Training curves
from IPython.display import Image, display
print("📈 Training Results:")
display(Image(filename=f"{run_dir}/results.png", width=800))


In [ ]:
# Confusion matrix
print("📊 Confusion Matrix:")
display(Image(filename=f"{run_dir}/confusion_matrix.png", width=500))


In [ ]:
# Validation metrics
best_model = YOLO(f"{run_dir}/weights/best.pt")
metrics = best_model.val(data=yaml_path, imgsz=IMG_SIZE, device=0)

print(f"\n{'='*50}")
print("  NETRA Model Evaluation")
print(f"{'='*50}")
print(f"  Box  mAP@50    : {metrics.box.map50:.4f}")
print(f"  Box  mAP@50-95 : {metrics.box.map:.4f}")
print(f"  Box  Precision : {metrics.box.mp:.4f}")
print(f"  Box  Recall    : {metrics.box.mr:.4f}")
if hasattr(metrics, "seg") and metrics.seg is not None:
    print(f"  ─────────────────────────────────────────")
    print(f"  Mask mAP@50    : {metrics.seg.map50:.4f}")
    print(f"  Mask mAP@50-95 : {metrics.seg.map:.4f}")
print(f"{'='*50}")


## Step 5: Download Model


In [ ]:
# Download best.pt to your PC
from google.colab import files

best_pt_path = f"{run_dir}/weights/best.pt"
print(f"📦 Model size: {os.path.getsize(best_pt_path) / 1e6:.1f} MB")
files.download(best_pt_path)

print("\n" + "="*50)
print("  DONE! Copy best.pt → NETRA/weights/best.pt")
print("  Then run: python demo.py")
print("="*50)


### Optional: Save to Google Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/NETRA"
os.makedirs(save_dir, exist_ok=True)
shutil.copy2(best_pt_path, f"{save_dir}/best.pt")
print(f"✅ Saved to Google Drive: {save_dir}/best.pt")
